In [1]:
!pip install pyclustering

     ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
     ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
     -------- ------------------------------- 0.5/2.6 MB 1.5 MB/s eta 0:00:02
     ------------ --------------------------- 0.8/2.6 MB 1.5 MB/s eta 0:00:02
     ---------------- ----------------------- 1.0/2.6 MB 1.4 MB/s eta 0:00:02
     -------------------- ------------------- 1.3/2.6 MB 1.4 MB/s eta 0:00:01
     ------------------------ --------------- 1.6/2.6 MB 1.4 MB/s eta 0:00:01
     ---------------------------- ----------- 1.8/2.6 MB 1.4 MB/s eta 0:00:01
     -------------------------------- ------- 2.1/2.6 MB 1.4 MB/s eta 0:00:01
     -------------------------------- ------- 2.1/2.6 MB 1.4 MB/s eta 0:00:01
     ------------------------------------ --- 2.4/2.6 MB 1.2 MB/s eta 0:00:01
     ---------------------------------------- 2.6/2.6 MB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): fini

  DEPRECATION: Building 'pyclustering' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pyclustering'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

from pyclustering.cluster.kmedoids import kmedoids
from pyclustering.utils import distance_metric, type_metric  

from sklearn import datasets
from sklearn.preprocessing import StandardScaler

In [5]:
wine = datasets.load_wine()

In [6]:
wine_df = pd.DataFrame(wine.data,columns = wine.feature_names)

In [7]:
wine_df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


In [8]:
wine_df.tail()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
173,13.71,5.65,2.45,20.5,95.0,1.68,0.61,0.52,1.06,7.7,0.64,1.74,740.0
174,13.40,3.91,2.48,23.0,102.0,1.80,0.75,0.43,1.41,7.3,0.70,1.56,750.0
175,13.27,4.28,2.26,20.0,120.0,1.59,0.69,0.43,1.35,10.2,0.59,1.56,835.0
176,13.17,2.59,2.37,20.0,120.0,1.65,0.68,0.53,1.46,9.3,0.60,1.62,840.0
177,14.13,4.10,2.74,24.5,96.0,2.05,0.76,0.56,1.35,9.2,0.61,1.60,560.0


In [9]:
wine_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   alcohol                       178 non-null    float64
 1   malic_acid                    178 non-null    float64
 2   ash                           178 non-null    float64
 3   alcalinity_of_ash             178 non-null    float64
 4   magnesium                     178 non-null    float64
 5   total_phenols                 178 non-null    float64
 6   flavanoids                    178 non-null    float64
 7   nonflavanoid_phenols          178 non-null    float64
 8   proanthocyanins               178 non-null    float64
 9   color_intensity               178 non-null    float64
 10  hue                           178 non-null    float64
 11  od280/od315_of_diluted_wines  178 non-null    float64
 12  proline                       178 non-null    float64
dtypes: fl

In [13]:
wine_df.describe()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
count,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000
mean,13.000618,2.336348,2.366517,19.494944,99.741573,2.295112,2.029270,0.361854,1.590899,5.058090,0.957449,2.611685,746.893258
std,0.811827,1.117146,0.274344,3.339564,14.282484,0.625851,0.998859,0.124453,0.572359,2.318286,0.228572,0.709990,314.907474
min,11.030000,0.740000,1.360000,10.600000,70.000000,0.980000,0.340000,0.130000,0.410000,1.280000,0.480000,1.270000,278.000000
25%,12.362500,1.602500,2.210000,17.200000,88.000000,1.742500,1.205000,0.270000,1.250000,3.220000,0.782500,1.937500,500.500000
50%,13.050000,1.865000,2.360000,19.500000,98.000000,2.355000,2.135000,0.340000,1.555000,4.690000,0.965000,2.780000,673.500000
75%,13.677500,3.082500,2.557500,21.500000,107.000000,2.800000,2.875000,0.437500,1.950000,6.200000,1.120000,3.170000,985.000000
max,14.830000,5.800000,3.230000,30.000000,162.000000,3.880000,5.080000,0.660000,3.580000,13.000000,1.710000,4.000000,1680.000000


In [11]:
scaler = StandardScaler()
x_scaler = scaler.fit_transform(wine.data)
x_scaler

array([[ 1.51861254, -0.5622498 ,  0.23205254, ...,  0.36217728,
         1.84791957,  1.01300893],
       [ 0.24628963, -0.49941338, -0.82799632, ...,  0.40605066,
         1.1134493 ,  0.96524152],
       [ 0.19687903,  0.02123125,  1.10933436, ...,  0.31830389,
         0.78858745,  1.39514818],
       ...,
       [ 0.33275817,  1.74474449, -0.38935541, ..., -1.61212515,
        -1.48544548,  0.28057537],
       [ 0.20923168,  0.22769377,  0.01273209, ..., -1.56825176,
        -1.40069891,  0.29649784],
       [ 1.39508604,  1.58316512,  1.36520822, ..., -1.52437837,
        -1.42894777, -0.59516041]], shape=(178, 13))

In [12]:
initial_medoids = [0,1,2]

In [16]:
import random
initial_medoids = random.sample(range(len(x_scaler)),3)

In [15]:
metric = distance_metric(type_metric.EUCLIDEAN)

kmedoids_instance = kmedoids(x_scaler,initial_medoids,metric = metric)

In [17]:
kmedoids_instance.process()

In [19]:
cluster = kmedoids_instance.get_clusters()
medoids = kmedoids_instance.get_medoids()

In [21]:
# plot the results
plt.figure(figsize=(8, 6))
for cluster in clusters:
    plt.scatter(X_scaled[cluster, 0], X_scaled[cluster, 1], label=f'Cluster {clusters.index(cluster)}')

# marks the medoids in red 

for medoid in medoids:
    plt.scatter(X_scaled[medoid, 0], X_scaled[medoid, 1], s=200, c='red', marker='X', label='Medoid')

plt.title('K-Medoids Clustering of (Wine Dataset) with PyClustering')
  

plt.legend()
plt.show()

NameError: name 'clusters' is not defined

<Figure size 800x600 with 0 Axes>